# 01 · Target Factory: the targets and the model input

**What it does.** It builds each use case's target from its target source, joins the feature
sources and saves the **model input**: the sources joined plus one `target_<use_case>` column per
use case. Every later stage reads it with its own config, keeps its own target as `target` and
derives eligibility, the candidates, the contract checks and the client split on read, so those
can change without a rebuild.

**The example** saves two use cases on the synthetic data, `cc_acquisition` (credit card) and
`td_acquisition` (term deposit); use cases saved together share the feature sources, keys and
period. The CLI: `python -m core_modelling run conf/base/cc_acquisition.yml
conf/base/td_acquisition.yml --stages target_factory`.

**Decisions:** none, at either automation level: the data definition is the config. A wrong target
means changing its config and saving again; a wrong filter or candidate list, changing the config
and rerunning from EDA.

**Docs:** `docs/components/target_factory.md` · `docs/components/dataset.md` · next: `02_eda`.

In [ ]:
from pathlib import Path

import pandas as pd

import core_modelling
from core_modelling.config import load_config
from core_modelling.dataset import build_frames, save_model_input
from core_modelling.environment import get_spark_session, load_environment
from core_modelling.target_factory import run_target_factory

# The use cases and where the framework runs, as absolute paths. The repo's examples are found
# from the installed package; a use-case repo writes its own config paths here and may read the
# environment file from $CORE_MODELLING_ENVIRONMENT instead.
REPO = Path(core_modelling.__file__).resolve().parents[2]
CONFIGS = [REPO / "conf/base/cc_acquisition.yml", REPO / "conf/base/td_acquisition.yml"]
ENTITY = "entity_a"
ENVIRONMENT = REPO / "conf/local/environment.yml"  # or None: $CORE_MODELLING_ENVIRONMENT

env = load_environment(str(ENVIRONMENT) if ENVIRONMENT else None)
spark = get_spark_session(env)
cfgs = [load_config(path) for path in CONFIGS]
for cfg in cfgs:
    print(cfg.use_case, cfg.task, cfg.data.target)
    print(cfg.time)

## 1. The targets
`run_target_factory` builds each use case's target on every row of its target source, before
eligibility, and prints a summary per period; it writes nothing.
- `null_share`: the rows whose target is unknown because their next `w` periods are incomplete
  (the last `w` periods, at 1.0; clients who leave; gaps).
- `null_value_share`: the complete windows whose target column held a null. A null never counts as
  an event, so a high share may hide events: have it filled upstream.
- `mean_target` (the rate), `positive_rows` and `positive_events` (distinct clients) cover the
  observation window only. In the `all` row a client is positive in up to `w` consecutive periods,
  so it has fewer events than positive rows.

Healthy: a plausible, steady rate and null shares of a few percent at most outside the last `w`
periods. Anything else is a question for the data owner before modelling.

In [ ]:
summaries = {cfg.use_case: run_target_factory(cfg, ENTITY, env, spark) for cfg in cfgs}

## 2. Save the model input
`save_model_input` joins the sources once and writes the table where the environment file points
`data.model_input`, with `_model_input.json` beside it (the sources, the periods covered, the join
log and each target's definition). Each `join` line gives the share of the spine's rows (the first
feature source's) that a source matched; a low share usually means a key or a period tag that
differs between sources. To add a use case later, save again with every config listed. The next
cell lists the target columns saved.

In [ ]:
path = save_model_input(cfgs, ENTITY, env, spark)

In [ ]:
[column for column in spark.read.parquet(path).columns if column.startswith("target_")]

## 3. What EDA will see, per use case
`build_frames` reads the table back as EDA will: each use case keeps its own target as `target`,
then applies its eligibility, contract checks and split, printing a line each (`eligibility`: an
analysis filter's rows in → out, a modelling filter's rows tagged 1; `contract`: checks on the
whole population). Each frame holds `unit_id`, `period`, the candidates, the modelling tags,
`eligible`, `target` and `split`.

The table compares the use cases. `cc_acquisition` has fewer rows: it drops the clients with
neither a known income nor a salary transfer. A filter that drops nothing or almost everything
usually has a wrong condition: fix the config, no rebuild needed. Both use cases' frames sit in
memory at once; with large use cases, build one at a time.

In [ ]:
frames = {cfg.use_case: build_frames(cfg, ENTITY, env, spark, stage="eda") for cfg in cfgs}
pd.DataFrame.from_dict(
    {
        use_case: {
            "snapshot period": f.meta["snapshot_period"],
            "snapshot rows": len(f.snapshot),
            "candidate columns": len(f.meta["candidates"]),
            "eligible share": f.snapshot["eligible"].mean(),
            "target mean (eligible)": f.snapshot.loc[f.snapshot["eligible"] == 1, "target"].mean(),
        }
        for use_case, f in frames.items()
    },
    orient="index",
)

One use case in detail (change `USE_CASE` for the other): the lineage columns, excluded from the
candidates because the target is built from them; the population per compared period, steady when
healthy (the label columns exist for the window periods only); and the share of the snapshot's
rows each modelling tag keeps.

In [ ]:
USE_CASE = "cc_acquisition"
snapshot = frames[USE_CASE].snapshot
print(USE_CASE, "lineage excluded:", frames[USE_CASE].meta["lineage"])
population = frames[USE_CASE].meta["population"].set_index("period")
population.assign(eligible_share=population["eligible_rows"] / population["rows"])

In [ ]:
snapshot[[*frames[USE_CASE].meta["tags"], "eligible"]].mean().to_frame("snapshot share")